### 读写文件

加载和保存张量

In [46]:
import torch
from torch import nn
from torch.nn import functional as F

In [47]:
x = torch.arange(4)
torch.save(x, 'x_file')

In [48]:
x2 = torch.load('x_file')
x2

tensor([0, 1, 2, 3])

In [49]:
y = torch.zeros(4)
y

tensor([0., 0., 0., 0.])

In [50]:
torch.save([x, y], 'x_files')
x2, y2 = torch.load('x_files')
x2, y2

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

In [51]:
mydict = {'x':x, 'y':y}
torch.save(mydict, 'mydict_file')
dict2 = torch.load('mydict_file')
dict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

加载和保存模型参数

In [52]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(8, 256)
        self.output = nn.Linear(256, 10)
    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))

X = torch.randn(2, 8)
net = MLP()
Y = net(X)

In [53]:
torch.save(net.state_dict(), 'mlp_params')

In [54]:
clone = MLP()
clone.load_state_dict(torch.load('mlp_params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=8, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [55]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

想在一个新的网络中使用之
前网络的前两层，该怎么做？

设计一个，第一个网络中有三层，分别为：2 * 10， 10 * 20， 20 * 4
第二个网络使用第一个网络的前两层，第三层为：20 * 5

In [56]:
class FirstNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 10)
        self.fc2 = nn.Linear(10, 20)
        self.fc3 = nn.Linear(20, 4)
    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x 

In [57]:
model = FirstNet()
X = torch.randn(10, 2)
model(X)
first_two_layers = nn.Sequential(
    model.fc1,
    model.fc2
)

In [58]:
torch.save(model.state_dict(), 'model_params')

In [59]:
model.load_state_dict(torch.load('model_params'))

<All keys matched successfully>

In [60]:
class SecondNet(nn.Module):
    def __init__(self, first_two_layers):
        super().__init__()
        self.first_two_layers = first_two_layers 
        self.fc2 = nn.Linear(20, 5) 
    def forward(self, x):
        x = self.first_two_layers(x)  # 使用第一个模型的前两层
        x = self.fc2(x)
        return x

In [65]:
new_model = SecondNet(first_two_layers)
# new_model.load_state_dict(torch.load('model_params')[: 2])
print(new_model)
torch.load('model_params')

SecondNet(
  (first_two_layers): Sequential(
    (0): Linear(in_features=2, out_features=10, bias=True)
    (1): Linear(in_features=10, out_features=20, bias=True)
  )
  (fc2): Linear(in_features=20, out_features=5, bias=True)
)


OrderedDict([('fc1.weight',
              tensor([[-0.6951,  0.0799],
                      [ 0.5860, -0.2934],
                      [ 0.5001, -0.0331],
                      [-0.3941, -0.1233],
                      [ 0.3146,  0.5284],
                      [ 0.1149, -0.6504],
                      [ 0.3711, -0.4494],
                      [ 0.0657,  0.3686],
                      [ 0.4128, -0.1316],
                      [-0.0658,  0.3543]])),
             ('fc1.bias',
              tensor([ 0.1852, -0.6998, -0.0645, -0.3968,  0.3585, -0.1618, -0.0022,  0.3344,
                      -0.4405,  0.0927])),
             ('fc2.weight',
              tensor([[-0.1130,  0.3086, -0.2126,  0.0400,  0.3135,  0.0358, -0.0103,  0.1386,
                        0.1647, -0.1804],
                      [ 0.0578,  0.1663, -0.0005,  0.1929,  0.2915,  0.1218, -0.0521, -0.1280,
                       -0.0677,  0.2290],
                      [ 0.2213, -0.0105,  0.0917, -0.1746, -0.2213, -0.3044,  0.0545

In [ ]:
new_model(X)

tensor([[ 0.0179, -0.0409,  0.0351,  0.1176, -0.4664],
        [ 0.1088,  0.0209,  0.1162,  0.2193, -0.6036],
        [-0.1191, -0.1036, -0.1101, -0.0401, -0.2328],
        [ 0.1345,  0.0188,  0.1539,  0.2509, -0.6597],
        [-0.0524, -0.0045, -0.0911,  0.0264, -0.2860],
        [-0.0795, -0.1369, -0.0294,  0.0132, -0.3457],
        [-0.0405,  0.0097, -0.0852,  0.0387, -0.2985],
        [-0.0867, -0.1500, -0.0296,  0.0064, -0.3421],
        [ 0.0075,  0.0779, -0.0691,  0.0870, -0.3395],
        [ 0.0682, -0.0076,  0.0807,  0.1741, -0.5432]],
       grad_fn=<AddmmBackward0>)

In [ ]:
model.fc1.weight.data == new_model.first_two_layers[0].weight.data

tensor([[True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True]])

In [ ]:
model.fc2.weight.data == new_model.first_two_layers[1].weight.data

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True],
        [True, True,

In [ ]:
model.fc1.weight.data[0, 0] = 100
print(model.fc1.weight.data[0, 0] == new_model.first_two_layers[1].weight.data[0, 0])

tensor(False)
